# 01. 기초: OPD reward와 delta signal

목표: teacher, student, teacher-base의 token probability로 기존 OPD reward와 OPD²의 delta signal을 직접 계산합니다.

실행 방법: 위에서부터 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. Toy token distribution 만들기

세 모델은 같은 context에서 다음 token 후보에 확률을 부여합니다. Teacher는 reasoning tuning을 받은 모델이고, teacher_base는 그 이전 checkpoint라고 가정합니다.

In [ ]:
import math


student = {
    "therefore": 0.08,
    "perhaps": 0.22,
    "multiply": 0.30,
    "add": 0.18,
    "answer": 0.22,
}

teacher_base = {
    "therefore": 0.10,
    "perhaps": 0.25,
    "multiply": 0.28,
    "add": 0.12,
    "answer": 0.25,
}

teacher = {
    "therefore": 0.24,
    "perhaps": 0.08,
    "multiply": 0.10,
    "add": 0.38,
    "answer": 0.20,
}


def check_distribution(name, distribution):
    total = sum(distribution.values())
    print(f"{name:12s} total={total:.2f}")


for name, distribution in [("student", student), ("teacher_base", teacher_base), ("teacher", teacher)]:
    check_distribution(name, distribution)

## 2. Reward 계산

기존 OPD는 teacher와 student를 비교합니다. Delta signal은 teacher와 teacher_base를 비교합니다.

In [ ]:
def log_prob(distribution, token):
    return math.log(distribution[token])


def opd_reward(token):
    return log_prob(teacher, token) - log_prob(student, token)


def delta_reward(token):
    return log_prob(teacher, token) - log_prob(teacher_base, token)


print("token | R_OPD | R_delta")
print("--- | --- | ---")
for token in student:
    print(f"{token:9s} | {opd_reward(token):6.3f} | {delta_reward(token):7.3f}")

## 3. 해석하기

`add`는 reasoning-tuned teacher가 base보다 훨씬 더 선호하게 된 token입니다. 반대로 `perhaps`는 base에서는 자연스럽지만 reasoning tuning 후에는 덜 선호되는 token입니다. Delta signal은 이런 변화 방향을 강조합니다.

In [ ]:
ranked_by_delta = sorted(student, key=delta_reward, reverse=True)
ranked_by_opd = sorted(student, key=opd_reward, reverse=True)

print("OPD가 가장 강화하는 token:", ranked_by_opd[:3])
print("Delta가 가장 강화하는 token:", ranked_by_delta[:3])
print("Delta가 가장 억제하는 token:", ranked_by_delta[-2:])